# Ejercicio 3 - Consultas directas sobre archivos Parquet

Exploracion inicial de los viajes de taxis amarillos y verdes de 2026 usando DuckDB
**sin importar los datos a una tabla**: todas las consultas leen los `.parquet` de
`data/raw/` directamente.

- Las consultas SQL estan en [`sql/03_exploracion_parquet.sql`](../sql/03_exploracion_parquet.sql); este notebook las ejecuta por nombre, de modo que el SQL documentado y el ejecutado son el mismo.
- La documentacion de cada consulta (objetivo, fuente, resultado y decision) esta en [`docs/ejercicio3_exploracion.md`](../docs/ejercicio3_exploracion.md).
- Requisito: haber descargado los datos con `python scripts/download_data.py` (Ejercicio 2).

In [1]:
import os
import sys
import time
from pathlib import Path

import duckdb
import pandas as pd

# Trabajar desde la raiz del proyecto para que las rutas data/raw/... funcionen
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "scripts"))
from run_sql import cargar_consultas

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

con = duckdb.connect()   # base en memoria: no se crea ninguna tabla
CONSULTAS = cargar_consultas(Path("sql/03_exploracion_parquet.sql"))

def ejecutar(nombre, filas=100):
    descripcion, sql = CONSULTAS[nombre]
    inicio = time.perf_counter()
    df = con.sql(sql).df()
    print(f"{nombre}: {descripcion}\n({len(df)} filas, {time.perf_counter() - inicio:.2f} s)")
    return df.head(filas)

print("DuckDB", con.sql("SELECT version()").fetchone()[0], "| consultas cargadas:", len(CONSULTAS))

DuckDB v1.5.5 | consultas cargadas: 19


## 3.1 Cantidad de archivos

In [2]:
ejecutar("q01_cantidad_archivos")

q01_cantidad_archivos: 3.1 Cantidad de archivos Parquet disponibles, por tipo de taxi y anio.
(2 filas, 0.13 s)


,tipo,anio,archivos,primer_mes,ultimo_mes
0,green,2026,8,2026-01,2026-08
1,yellow,2026,8,2026-01,2026-08


## 3.2 Cantidad de registros

Primero solo con los metadatos de cada archivo (el *footer* del Parquet guarda la cantidad
de filas, asi que DuckDB no necesita leer los datos) y luego con un `count(*)` sobre los
datos para confirmar.

In [3]:
ejecutar("q02_registros_por_archivo")

q02_registros_por_archivo: 3.2 Registros por archivo leyendo solo los metadatos (footer) de cada Parquet.
(16 filas, 0.18 s)


,tipo,mes,registros,row_groups
0,green,2026-01,40272,1
1,green,2026-02,37373,1
2,green,2026-03,44208,1
3,green,2026-04,44238,1
4,green,2026-05,44921,1
5,green,2026-06,44163,1
6,green,2026-07,41252,1
7,green,2026-08,40687,1
8,yellow,2026-01,3724889,4
9,yellow,2026-02,3399866,4


In [4]:
ejecutar("q03_total_registros")

q03_total_registros: 3.2 Total de registros por tipo leyendo los datos (valida el resultado de q02).
(3 filas, 3.33 s)


,tipo,registros
0,green,337114
1,yellow,29703355
2,TOTAL,30040469


## 3.3 / 3.4 Columnas y tipos de datos

Antes de describir las columnas se revisa si el esquema es igual en todos los archivos.

In [5]:
ejecutar("q04_consistencia_esquema")

q04_consistencia_esquema: 3.3 / 3.4 Columnas que no aparecen en todos los archivos de su tipo, o que cambian de tipo entre archivos (deriva de esquema).
(2 filas, 0.21 s)


,tipo,columna,archivos_con_columna,archivos_del_tipo,tipos_vistos,meses
0,green,request_source,3,8,[BYTE_ARRAY StringType()],"[2026-06, 2026-07, 2026-08]"
1,yellow,request_source,3,8,[BYTE_ARRAY StringType()],"[2026-06, 2026-07, 2026-08]"


`request_source` solo existe a partir de junio de 2026. Esto tiene una consecuencia practica:
si se lee el glob **sin** `union_by_name`, DuckDB usa el esquema del primer archivo y la
columna desaparece sin ningun error:

In [6]:
sin_union = con.sql("DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet')").df()
con_union = con.sql("DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)").df()
print("columnas sin union_by_name:", len(sin_union), "| con union_by_name:", len(con_union))
print("perdidas:", sorted(set(con_union.column_name) - set(sin_union.column_name)))

columnas sin union_by_name: 20 | con union_by_name: 21
perdidas: ['request_source']


In [7]:
ejecutar("q05_columnas_yellow")

q05_columnas_yellow: 3.3 / 3.4 Columnas y tipos de datos de los taxis amarillos.
(21 filas, 0.09 s)


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [8]:
ejecutar("q06_columnas_green")

q06_columnas_green: 3.3 / 3.4 Columnas y tipos de datos de los taxis verdes.
(22 filas, 0.10 s)


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [9]:
ejecutar("q07_comparacion_columnas")

q07_comparacion_columnas: 3.3 Columnas comunes y exclusivas de cada tipo de taxi.
(25 filas, 0.16 s)


,columna,tipo_yellow,tipo_green,presente_en
0,DOLocationID,INTEGER,INTEGER,ambos
1,PULocationID,INTEGER,INTEGER,ambos
2,RatecodeID,BIGINT,BIGINT,ambos
3,VendorID,INTEGER,INTEGER,ambos
4,cbd_congestion_fee,DOUBLE,DOUBLE,ambos
5,congestion_surcharge,DOUBLE,DOUBLE,ambos
6,extra,DOUBLE,DOUBLE,ambos
7,fare_amount,DOUBLE,DOUBLE,ambos
8,improvement_surcharge,DOUBLE,DOUBLE,ambos
9,mta_tax,DOUBLE,DOUBLE,ambos


## 3.5 Muestra de registros

In [10]:
ejecutar("q08_muestra_yellow")

q08_muestra_yellow: 3.5 Muestra aleatoria reproducible de taxis amarillos.
(10 filas, 0.48 s)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-01-01 04:31:37,2026-01-01 04:36:45,3,1.01,1,N,246,48,1,7.2,1.00,0.5,2.59,0.00,1.0,15.54,2.5,0.00,0.75,None
1,2,2026-01-01 07:47:21,2026-01-01 07:49:32,1,0.39,1,N,141,141,1,4.4,0.00,0.5,1.00,0.00,1.0,9.40,2.5,0.00,0.00,None
2,2,2026-01-01 10:21:39,2026-01-01 10:24:35,1,0.72,1,N,50,246,1,5.8,0.00,0.5,2.64,0.00,1.0,13.19,2.5,0.00,0.75,None
3,1,2026-01-01 11:58:50,2026-01-01 12:07:08,1,1.70,1,N,239,161,1,10.0,3.25,0.5,2.95,0.00,1.0,17.70,2.5,0.00,0.75,None
4,2,2026-01-01 12:39:27,2026-01-01 12:52:43,1,4.79,1,N,140,232,1,21.2,0.00,0.5,5.19,0.00,1.0,31.14,2.5,0.00,0.75,None
5,2,2026-01-01 13:42:32,2026-01-01 13:54:47,1,1.37,1,N,162,68,1,12.1,0.00,0.5,3.37,0.00,1.0,20.22,2.5,0.00,0.75,None
6,2,2026-01-01 13:28:55,2026-01-01 13:37:15,3,1.43,1,N,100,164,1,9.3,0.00,0.5,2.81,0.00,1.0,16.86,2.5,0.00,0.75,None
7,2,2026-01-01 14:04:13,2026-01-01 14:27:47,4,10.19,1,N,100,138,1,42.2,5.00,0.5,2.00,6.94,1.0,60.89,2.5,0.00,0.75,None
8,1,2026-01-01 15:35:24,2026-01-01 16:07:48,1,22.00,1,N,132,60,1,80.7,0.00,0.5,22.30,6.94,1.0,111.44,0.0,0.00,0.00,None
9,2,2026-01-01 15:40:44,2026-01-01 16:26:18,1,19.12,2,N,132,79,1,70.0,0.00,0.5,5.00,0.00,1.0,81.50,2.5,1.75,0.75,None


In [11]:
ejecutar("q09_muestra_green")

q09_muestra_green: 3.5 Muestra aleatoria reproducible de taxis verdes.
(10 filas, 0.25 s)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-03 10:31:59,2026-01-03 10:35:48,N,1,75,263,1,0.63,5.80,0.0,0.5,0.00,0.0,NaN,1.0,10.05,2,1,2.75,0.0,None
1,2,2026-01-16 19:55:39,2026-01-16 20:12:48,N,1,97,61,1,3.06,18.40,1.0,0.5,0.00,0.0,NaN,1.0,20.90,2,1,0.00,0.0,None
2,2,2026-01-19 13:09:08,2026-01-19 13:29:44,N,1,65,225,1,2.98,19.80,0.0,0.5,5.00,0.0,NaN,1.0,26.30,1,1,0.00,0.0,None
3,2,2026-01-21 17:01:53,2026-01-21 17:10:44,N,1,43,263,2,0.65,9.30,2.5,0.5,2.66,0.0,NaN,1.0,15.96,1,1,0.00,0.0,None
4,2,2026-01-23 16:43:58,2026-01-23 16:57:21,N,1,223,138,1,3.53,17.70,7.5,0.5,5.34,0.0,NaN,1.0,32.04,1,1,0.00,0.0,None
5,2,2026-01-27 13:42:53,2026-01-27 14:03:40,N,1,74,151,1,2.62,19.10,0.0,0.5,5.15,0.0,NaN,1.0,25.75,1,1,0.00,0.0,None
6,1,2026-01-30 07:39:44,2026-01-30 07:45:09,N,1,74,41,0,0.60,6.50,0.0,1.5,0.00,0.0,NaN,1.0,8.00,2,1,0.00,0.0,None
7,2,2026-01-30 23:18:59,2026-01-30 23:30:41,N,1,97,66,1,1.37,11.40,1.0,0.5,2.08,0.0,NaN,1.0,15.98,1,1,0.00,0.0,None
8,2,2026-01-06 16:11:00,2026-01-06 16:35:00,NaN,<NA>,159,151,<NA>,3.16,28.59,0.0,0.5,0.00,0.0,NaN,1.0,30.09,<NA>,<NA>,NaN,0.0,None
9,6,2026-01-22 15:42:50,2026-01-22 16:03:37,NaN,<NA>,130,218,<NA>,3.23,3.00,0.0,0.5,0.00,0.0,NaN,0.3,17.26,<NA>,<NA>,NaN,0.0,None


## 3.6 Problemas de calidad de datos

### Valores nulos

In [12]:
ejecutar("q10_nulos_por_columna")

q10_nulos_por_columna: 3.6 Porcentaje de nulos por columna y tipo de taxi.
(14 filas, 55.26 s)


,tipo,columna,pct_nulos
0,yellow,request_source,90.23
1,yellow,Airport_fee,25.98
2,yellow,RatecodeID,25.98
3,yellow,congestion_surcharge,25.98
4,yellow,passenger_count,25.98
5,yellow,store_and_fwd_flag,25.98
6,green,ehail_fee,100.00
7,green,request_source,94.30
8,green,RatecodeID,14.47
9,green,congestion_surcharge,14.47


In [13]:
ejecutar("q11_nulos_simultaneos")

q11_nulos_simultaneos: 3.6 Los nulos de passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge (y Airport_fee) ocurren en las mismas filas?
(2 filas, 3.95 s)


,tipo,registros,passenger_count_nulo,todos_nulos_juntos,nulos_con_payment_0,payment_type_0
0,yellow,29703355,7716688,7716688,7716688,7716688
1,green,337114,48775,48775,0,0


### Fechas fuera del periodo del archivo

In [14]:
ejecutar("q12_fechas_fuera_de_periodo")

q12_fechas_fuera_de_periodo: 3.6 Viajes cuya fecha de inicio no corresponde al mes del archivo.
(2 filas, 3.94 s)


,tipo,registros,fuera_del_mes,mas_de_un_mes_antes,despues_del_mes,pickup_minimo,pickup_maximo
0,yellow,29703355,146,26,47,2001-01-01 09:23:58,2026-08-31 23:59:59
1,green,337114,98,10,29,2008-12-31 17:35:31,2026-08-31 23:58:28


### Duracion y distancia

In [15]:
ejecutar("q13_duracion_viajes")

q13_duracion_viajes: 3.6 Duraciones imposibles o sospechosas.
(2 filas, 7.49 s)


,tipo,registros,dropoff_antes_de_pickup,duracion_cero,menos_de_1_min,mas_de_3_horas,mas_de_24_horas,mediana_min,p99_min,max_min
0,yellow,29703355,10,371673,309479,10127,263,13.93,71.17,17165.4
1,green,337114,5,229,11042,1287,4,13.07,82.23,2456.4


In [16]:
ejecutar("q14_distancias")

q14_distancias: 3.6 Distancias en cero o extremas (trip_distance esta en millas).
(2 filas, 6.09 s)


,tipo,registros,distancia_cero,mas_de_100_millas,mas_de_1000_millas,mediana_millas,p99_millas,max_millas
0,yellow,29703355,952231,1223,615,1.86,19.50,328522.20
1,green,337114,12212,72,66,2.07,17.76,179830.92


### Montos

In [17]:
ejecutar("q15_montos")

q15_montos: 3.6 Montos negativos, en cero, extremos e inconsistentes con sus componentes.
(2 filas, 6.45 s)


,tipo,registros,tarifa_negativa,total_negativo,total_cero,propina_negativa,total_mayor_500,total_no_cuadra,pct_no_cuadra
0,yellow,29703355,157364,161835,5258,883,791,10895756,36.68
1,green,337114,999,1023,543,69,21,66970,19.87


In [18]:
ejecutar("q15b_montos_por_proveedor")

q15b_montos_por_proveedor: 3.6 Diferencia entre total_amount y la suma de componentes, por proveedor y forma de pago. Sirve para saber si el total esta mal o si cada proveedor codifica los recargos de forma distinta.
(22 filas, 6.99 s)


,tipo,VendorID,payment_type,registros,no_cuadra,pct_no_cuadra,diferencia_mas_comun
0,yellow,1,0,895381,881364,98.4,2.50
1,yellow,1,1,4028246,3315691,82.3,-3.25
2,yellow,1,2,447521,416904,93.2,-3.25
3,yellow,1,3,59767,50793,85.0,-3.25
4,yellow,1,4,36154,28447,78.7,-3.25
5,yellow,2,0,6761917,5963133,88.2,2.50
6,yellow,2,1,14592874,16620,0.1,2.50
7,yellow,2,2,2217837,5381,0.2,2.50
8,yellow,2,3,35455,70,0.2,-2.50
9,yellow,2,4,201691,416,0.2,-2.50


### Codigos categoricos, zonas y duplicados

In [19]:
ejecutar("q16_codigos_categoricos")

q16_codigos_categoricos: 3.6 Distribucion de las columnas codificadas, para detectar codigos fuera del diccionario de datos de la TLC.
(70 filas, 11.93 s)


,tipo,columna,valor,registros
0,yellow,RatecodeID,1,20102072
1,yellow,RatecodeID,NaN,7716688
2,yellow,RatecodeID,99,769693
3,yellow,RatecodeID,2,694936
4,yellow,RatecodeID,5,262614
...,...,...,...,...
65,green,request_source,HV0005,1179
66,green,request_source,CC,3
67,green,trip_type,1,273386
68,green,trip_type,NaN,48777


In [20]:
ejecutar("q17_zonas_desconocidas")

q17_zonas_desconocidas: 3.6 Viajes con zona de origen o destino 264/265 (zonas "Unknown"/"Outside of NYC" en la tabla de zonas de la TLC).
(2 filas, 2.96 s)


,tipo,registros,origen_264_265,destino_264_265,pct_afectado
0,yellow,29703355,49676,178136,0.68
1,green,337114,1131,5600,1.75


In [21]:
ejecutar("q18_duplicados")

q18_duplicados: 3.6 Registros exactamente duplicados (todas las columnas iguales). Es la consulta mas costosa del ejercicio (~1 min): DISTINCT sobre ~30 M de filas.
(2 filas, 36.33 s)


,tipo,registros,filas_distintas,duplicados
0,yellow,29703355,29703348,7
1,green,337114,337114,0


## 3.9 Que significa consultar directamente un Parquet

El plan de ejecucion muestra que DuckDB lee los archivos con `READ_PARQUET` dentro de la
consulta, sin una tabla intermedia, y que solo proyecta las columnas que la consulta usa
(*projection pushdown*). Con un filtro, ademas, puede descartar row groups completos usando
las estadisticas min/max guardadas en el archivo (*filter pushdown*).

In [22]:
plan = con.sql('''
    EXPLAIN ANALYZE
    SELECT payment_type, avg(tip_amount)
    FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)
    WHERE tpep_pickup_datetime >= TIMESTAMP '2026-08-01'
    GROUP BY payment_type
''').fetchall()[0][1]
print(plan)

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE     SELECT payment_type, avg(tip_amount)     FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true)     WHERE tpep_pickup_datetime >= TIMESTAMP '2026-08-01'     GROUP BY payment_type 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.306s              ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────